# Imports & Functions

In [1]:
import pandas as pd
from pathlib import Path
from ydata_profiling import ProfileReport
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from importlib import reload
import data_loading  # Import the module instead of specific functions


reload(data_loading)  # Reload the module after making changes so that the kernel resets

# reference functions directly from the reloaded module
read_csv_to_dataframe = data_loading.read_csv_to_dataframe
read_txt_to_dataframe = data_loading.read_txt_to_dataframe

/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""
Expands the categories in a given column of a DataFrame into separate binary columns.

Parameters:
- df: pandas.DataFrame, the DataFrame to modify.
- column_name: str, the name of the column to expand.
"""
def expand_categories_in_column(df, column_name):

    # create set of unique entries from a string based on splitting at ", "
    unique_categories = set()
    df[column_name].dropna().apply(lambda x: unique_categories.update(set(x.split(', '))))

    # init dict to hold new columns
    new_cols = {}
    
    for category in unique_categories:
        # add col name as there are duplicates across different cols
        # remove spaces and commas from col name
        valid_category_name = column_name + " / " + category.lower().replace(' ', '_').replace(',', '')
        
        # Instead of modifying df directly, create and store the new column in new_cols
        mask = df[column_name].fillna('').str.contains(category, regex=False, na=False)
        new_cols[valid_category_name] = mask.astype(int)

    # Create a new df from the new_cols dictionary
    new_columns_df = pd.DataFrame(new_cols, index=df.index)
    
    # Concatenate the new columns to the original DataFrame
    df = pd.concat([df, new_columns_df], axis=1)

    return df

In [3]:
## preprocessing script for logistic regressions dataframes
## based on test_df
def preprocess_df(test_df, unique_col):

    # make a copy, take the appropriate cols, expand accordingly
    df_copy = test_df.copy()
    df_copy = df_copy[['substance carried', unique_col]]
    print(df_copy.count())
    
    df_copy[unique_col] = df_copy[unique_col].str.lower()
    df_copy = expand_categories_in_column(df_copy, unique_col)
    df_copy.drop(columns=[unique_col], inplace=True)
    
    return df_copy

In [4]:
# correlation matrix and dropping highly-correlated pairs for logistic regression
def handle_correlation_and_drop_columns(test_df):
    # Correlation matrix and identify pairs with correlation > 0.8 (and less than 1)
    corr_matrix = test_df.corr()
    high_corr_pairs = corr_matrix.unstack().sort_values(kind="quicksort", ascending=False)
    high_corr_pairs = high_corr_pairs[(abs(high_corr_pairs) > 0.8) & (high_corr_pairs != 1)]
    
    # Identify columns to drop based on correlations > 0.8
    threshold = 0.8
    to_drop = set()
    for (col1, col2), corr in high_corr_pairs.items():
        if corr > threshold:
            # drop col with less entries
            sum_col1 = test_df[col1].sum()
            sum_col2 = test_df[col2].sum()
            if sum_col1 < sum_col2:
                to_drop.add(col1)
            else:
                to_drop.add(col2)
    
    # Drop cols from above from dataframe
    reduced_df = test_df.drop(columns=list(to_drop))
    return reduced_df

In [5]:
# logistic regression for qualitative cols
def fit_and_evaluate_logistic_regression(test_df):
    # 'substance carried' is the target variable
    X = test_df.drop(['substance carried'], axis=1)  # Features
    y = test_df['substance carried']  # Target
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    
    # Create and fit the logistic regression model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    # Predictions and evaluation
    y_pred = model.predict(X_test)
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred), "\n")
    print("Classification report:\n", classification_report(y_test, y_pred), "\n")
    
    # Coefficients
    coefficients = pd.DataFrame(model.coef_.flatten(), index=X.columns, columns=['Coefficient'])
    sorted_coefficients = coefficients.sort_values(by='Coefficient', ascending=False)
    print("Sorted coefficients:\n", sorted_coefficients, "\n")
    
    return model

# Load in dataset A

In [6]:
## making file path imports more robust

# get directory of current file
current_script_directory = Path.cwd()

# Construct path to data files given relative location
cad_string = current_script_directory / "../data/raw/canada/"
cad_data = cad_string / "pipeline-incidents-comprehensive-data.csv"

# read data into dataframes
CAD_data_raw = read_csv_to_dataframe(cad_data)

File at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/canada/pipeline-incidents-comprehensive-data.csv' successfully read into a DataFrame.


# Data Cleaning

In [7]:
# make deep copy for cleaned df
CAD_data_cleaned = CAD_data_raw.copy()

### Data Dictionary

In [8]:
# looked at data dictionary for columns that could be relevant to substance / pipeline specifications / cause of incident
# want to analyze this subset for trends/correlations
## data dict and data don't line up so needed to manually change some; labelled with "## different"
subset_columns = [
"pipeline or facility type",
"pipeline or facility equipment involved",
"rupture",
"incident types", ## different
"conditions that resulted in the operation beyond limits",
"pipeline outside diameter (nps)",
"pipeline length (km)",
"substance carried",
"released substance type",
"facility type", ## different
"facility latitude",
"facility longitude",
"longitude",
"latitude",
"nominal pipe size",
"material",
"material grade",
"schedule",
"design wall thickness (mm)",
"custom design wall thickness (mm)",
"actual wall thickness (mm)",
"licensed maximum operating pressure (kpa)",
"actual operating pressure at time of failure (kpa)",
"year of manufacture",
"most recent cathodic protection reading at incident site (mv vs. cu/cuso4)",
"weld type",
"seam type",
"coating location",
"coating type",
"coating condition",
"application method",
"year when the coating was applied",
"insulation installed",
"detailed what happened", ## diff
"what happened category", ## diff
"detailed why it happened", ## diff
"why it happened category" ## diff
]

# Taking the subset
CAD_data_cleaned = CAD_data_cleaned[subset_columns]

# only going to take first of pipeline outside diameter values
CAD_data_cleaned['pipeline outside diameter (nps)'] = CAD_data_cleaned['pipeline outside diameter (nps)'].str.split(',').str[0].astype(float).fillna(0)

# fill empty length values with 0
CAD_data_cleaned['pipeline length (km)'] = CAD_data_cleaned['pipeline length (km)'].astype(float).fillna(0)

# removing inches and keeping in mm
CAD_data_cleaned['design wall thickness (mm)'] = CAD_data_cleaned['design wall thickness (mm)'].str.split(' mm').str[0].astype(float).fillna(0)

## 3 substance-based columns - if they're all empty/NA then we'll drop the rows as these don't really help us
## EDIT: "substance" and "released substance type" are identical cols so removed substance
CAD_data_cleaned = CAD_data_cleaned.loc[~((CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].isna()))]

### Substance carried / released substance

In [9]:
## lots of na's in substance carried col - try to imputate data as best as we can

## OPTION 1: substance carried contains crude oil, released substance does not; released substance = lube oil, drilling fluid, natural gas liquids, diesel fuel, condensate
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')==False)][['released substance type', 'substance carried']].drop_duplicates()


,released substance type,substance carried
153,Lube Oil,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
232,Drilling Fluid,Crude Oil
284,Natural Gas Liquids,Crude Oil
349,Natural Gas Liquids,"Condensate, Crude Oil, Natural Gas Liquids"
593,Drilling Fluid,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
1050,Natural Gas - Sweet,Crude Oil
1160,Diesel Fuel,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
1487,Condensate,Crude Oil
1838,Hydraulic Fluid,Crude Oil


In [10]:
## OPTION 2: substance carried contains crude oil, released subsance does too
## OBSERVATION: released substance type sometimes is more specific (sour vs. sweet) than substance carried
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil'))][['released substance type', 'substance carried']].drop_duplicates()


,released substance type,substance carried
10,Crude Oil - Sour,Crude Oil
29,Crude Oil - Sweet,Crude Oil
183,Crude Oil - Synthetic,Crude Oil
196,Crude Oil - Sweet,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."


In [11]:
## OPTION 3: substance carried doesn't contain crude oil, released substance does? 
## NO RESULTS
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')==False) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil'))][['released substance type', 'substance carried']].drop_duplicates()

,released substance type,substance carried


In [12]:
## OPTION 4: substance carried doesn't contain crude oil, released substance doesn't either
## HAPPENS A LOT - and the released substance type looks similar as if the substance carried was crude oil
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')==False) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')==False)][['released substance type', 'substance carried']].drop_duplicates()

,released substance type,substance carried
4,Natural Gas - Sweet,Natural Gas
14,Jet Fuel,"not applicable, Refined Products-Aviation, Ref..."
16,Natural Gas - Sweet,"Natural Gas, Natural Gas Sweet"
17,Water,White Water
31,Natural Gas - Sour,Natural Gas Sour
79,Natural Gas - Sweet,Natural Gas Sweet
87,Pulp slurry,Sulfite Pulp Slurry
101,Mixed HVP Hydrocarbons,Natural Gas Sweet
281,Contaminated Water,"Natural Gas Sour, Natural Gas Sweet, not appli..."
282,Propane,Natural Gas Sour


In [13]:
# option 5 - substance carried is na, but released substance contains crude oil
## Can confidently change the substance carried to crude oil based on results from option #3
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil'))][['released substance type', 'substance carried']].drop_duplicates()
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')), 'substance carried'] = 'Crude Oil'

In [14]:
# option 6 - substance carried is na, released substance does not contain crude oil
## change substance carried to NOT crude oil IFF the released substance type has 0% change of appearing in crude oil (source of list: chatgpt)
not_in_pipeline_crude_oil = [
    "Potassium Hydroxide (caustic solution)",
    "Sulphur Dioxide",
    "Water",
    "Potassium Carbonate",
    "Contaminated Water",
    "Waste Oil",
    "Amine",
    "Produced Water",
    "Glycol",
    "Pulp slurry"
]
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')==False)][['released substance type', 'substance carried']].drop_duplicates()
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].isin(not_in_pipeline_crude_oil)), 'substance carried'] = 'NOT crude oil'



In [15]:
# OPTION 7/8 - released substance type is na; can't assume what was released (if anything); ignore! 
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')) & (CAD_data_cleaned['released substance type'].isna())][['released substance type', 'substance carried']].drop_duplicates()

,released substance type,substance carried
22,NaN,Crude Oil
24,NaN,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
1732,NaN,"Condensate, Crude Oil"


In [16]:
# still left with lots of records where substance carried is N/A
# sadly have to drop these as it's going to be our target/key attribute and we cannot imputate any further
print("dropping {} rows as they don't have a substance carried".format(len(CAD_data_cleaned.loc[CAD_data_cleaned['substance carried'].isna()])))
CAD_data_cleaned = CAD_data_cleaned.loc[CAD_data_cleaned['substance carried'].isna()==False]

dropping 396 rows as they don't have a substance carried


In [17]:
# # drop columns that have more than x percentage of na's
# # can play around with this value to see how it affects results
# threshold = 0.5
# CAD_data_cleaned = CAD_data_cleaned.loc[:, CAD_data_cleaned.isnull().mean() < 0.5]

# #categorical_cols = CAD_data_cleaned.select_dtypes(include=['object']).columns
# #categorical_cols

# Crude vs. Not Crude

### Data Exploration

In [18]:
## want to update substance type so that it's either "Crude" or "not crude" so that we can more easily identify trends
all_cad = CAD_data_cleaned.copy()

# map all non-crude oil substances into one group to see if there's a difference
all_cad.loc[all_cad['substance carried'].astype(str).str.contains('Crude Oil')==False, 'substance carried'] = "NOT Crude Oil"

## FOR argument's sake, let's also combine the crude oils
all_cad.loc[all_cad['substance carried'].astype(str).str.contains("NOT Crude Oil")==False, 'substance carried'] = "Crude Oil"

In [19]:
## GENERATE a profiling report to identify any preliminary trends/issues in the data

# Define the path to the report file
report_path = Path("../reports/dataset_A_all_cad_profile_report.html")

# only generate report if it doesn't already exist
if not report_path.exists():
    all_cad_profile = ProfileReport(all_cad, title="All CAD data Profiling Report", explorative=True)
    all_cad_profile.to_file(report_path)
else:
    print("Report already exists.")

Report already exists.


In [20]:
# report generated duplicate rows and coating location has constant value
## DUPLICATES: after looking at data, the only variation in the duplicate rows is the incident number, so we can safely assume that they are duplicate rows. Drop.
## coating location - drop
print("dropping {} duplicate rows".format(len(all_cad[all_cad.duplicated()])))
all_cad = all_cad.drop_duplicates()
all_cad = all_cad.drop(columns={'coating location'})

dropping 49 duplicate rows


In [21]:
# SPLIT df into substance carried contains crude oil or does not so that we can get quantitative descriptive statistics (qualitative in html file)
crude_cad = all_cad.copy()
crude_cad = crude_cad.loc[crude_cad['substance carried'] == "Crude Oil"]

non_crude_cad = all_cad.copy()
non_crude_cad = non_crude_cad.loc[non_crude_cad['substance carried'] == "NOT Crude Oil"]

In [22]:
non_crude_cad.describe()

,pipeline outside diameter (nps),pipeline length (km),facility latitude,facility longitude,longitude,latitude,schedule,design wall thickness (mm),custom design wall thickness (mm),actual wall thickness (mm),licensed maximum operating pressure (kpa),actual operating pressure at time of failure (kpa),year of manufacture,most recent cathodic protection reading at incident site (mv vs. cu/cuso4),year when the coating was applied
count,686.000000,686.000000,151.000000,151.000000,686.000000,686.000000,45.000000,686.000000,34.000000,59.000000,203.000000,113.000000,67.000000,5.800000e+01,61.000000
mean,503.412536,565.185590,51.604011,-104.765424,-103.484240,51.572822,50.222222,1.584111,257.794118,7.718644,6691.526015,4957.716124,1972.537313,3.212361e+07,1970.622951
std,331.500516,894.587596,4.612020,19.511114,19.893285,4.441162,27.009164,3.343502,1447.648429,14.477015,2618.791973,2362.952651,18.769195,2.446502e+08,12.845445
min,0.000000,0.000000,42.841123,-122.704200,-122.726499,42.287869,20.000000,0.000000,0.000000,0.900000,0.000000,0.000000,1950.000000,-2.124000e+03,1950.000000
25%,273.100000,19.530748,48.551335,-119.922662,-119.778867,47.914485,30.000000,0.000000,3.900000,3.600000,6387.500000,3945.000000,1957.000000,-1.273500e+03,1962.000000
50%,508.000000,104.621705,51.943617,-114.117508,-112.656655,52.592710,40.000000,0.000000,6.100000,4.800000,6895.000000,5290.000000,1970.000000,-9.200000e+02,1971.000000
75%,762.000000,635.018816,55.721482,-82.062131,-80.126535,55.572706,80.000000,0.000000,9.475000,7.950000,8412.000000,6753.000000,1979.000000,-1.084000e+00,1975.000000
max,1362.400000,3393.426420,58.654722,-61.612676,-61.612676,59.201700,160.000000,19.700000,8450.000000,114.000000,14400.000000,11721.000000,2019.000000,1.863200e+09,2017.000000


In [23]:
# OBSERVATION: large difference between licensend maximum kpa and actual operating kpa at time of failure for crude
# different in non crude is smaller. but this is  SMALL sample size
crude_cad.describe()

,pipeline outside diameter (nps),pipeline length (km),facility latitude,facility longitude,longitude,latitude,schedule,design wall thickness (mm),custom design wall thickness (mm),actual wall thickness (mm),licensed maximum operating pressure (kpa),actual operating pressure at time of failure (kpa),year of manufacture,most recent cathodic protection reading at incident site (mv vs. cu/cuso4),year when the coating was applied
count,376.000000,376.000000,98.000000,98.000000,376.000000,376.000000,11.000000,376.000000,7.000000,19.000000,52.000000,29.000000,25.000000,13.000000,18.000000
mean,642.605319,946.347810,50.589189,-114.784673,-113.632863,50.680141,76.363636,0.781915,7.800000,7.663158,6249.305769,3261.529483,1978.960000,-385.291769,1963.722222
std,292.541820,569.769831,2.708496,11.676238,11.879089,2.831448,50.452498,2.394971,1.138713,2.547376,10432.059819,2595.043028,27.006913,1136.132885,18.833030
min,0.000000,0.000000,42.952615,-122.950276,-123.702550,42.834210,20.000000,0.000000,5.600000,1.700000,0.000000,0.000000,1950.000000,-1169.000000,1950.000000
25%,584.200000,946.159155,49.268489,-122.931342,-121.688203,49.261809,40.000000,0.000000,7.900000,7.100000,1253.250000,892.000000,1952.000000,-1056.000000,1952.000000
50%,762.000000,946.159155,50.542893,-119.307137,-119.249766,50.242489,80.000000,0.000000,7.900000,7.900000,6624.000000,3000.000000,1972.000000,-955.000000,1953.000000
75%,762.000000,1255.999121,52.643637,-111.278646,-111.165757,52.639096,100.000000,0.000000,7.900000,8.750000,7470.250000,5737.000000,2009.000000,-1.286000,1968.750000
max,1219.000000,2334.948756,59.066738,-72.627000,-72.478000,63.668419,160.000000,12.700000,9.500000,12.400000,75403.000000,7939.000000,2020.000000,3033.000000,2014.000000


### Logistic Regression

In [24]:
all_cad.groupby(['substance carried'])[['what happened category', 'detailed what happened', 'incident types']].count()

,what happened category,detailed what happened,incident types
substance carried,,,
Crude Oil,376,367,376
NOT Crude Oil,686,679,686


In [25]:
# make test df
test_df = all_cad.copy()
test_df['substance carried'] = test_df['substance carried'].map({"NOT Crude Oil": 0, "Crude Oil": 1})

# preprocess test df into 3 dfs each with the respective categorical col expanded
test_df_1 = preprocess_df(test_df, 'what happened category')
test_df_2 = preprocess_df(test_df, 'detailed what happened')
test_df_3 = preprocess_df(test_df, 'incident types')

substance carried         1062
what happened category    1062
dtype: int64
substance carried         1062
detailed what happened    1046
dtype: int64
substance carried    1062
incident types       1062
dtype: int64


In [26]:
# correlation/preprocessing
test_df_1_reduced = handle_correlation_and_drop_columns(test_df_1)
test_df_2_reduced = handle_correlation_and_drop_columns(test_df_2)
test_df_3_reduced = handle_correlation_and_drop_columns(test_df_3)

# run regression model on each df
print("Results for 'what happened category'")
model_1 = fit_and_evaluate_logistic_regression(test_df_1_reduced)

print("Results for 'detailed what happened'")
model_2 = fit_and_evaluate_logistic_regression(test_df_2_reduced)

print("Results for 'incident types'")
model_3 = fit_and_evaluate_logistic_regression(test_df_3_reduced)

Results for 'what happened category'
Confusion matrix:
 [[149  37]
 [ 43  37]] 

Classification report:
               precision    recall  f1-score   support

           0       0.78      0.80      0.79       186
           1       0.50      0.46      0.48        80

    accuracy                           0.70       266
   macro avg       0.64      0.63      0.63       266
weighted avg       0.69      0.70      0.70       266
 

Sorted coefficients:
                                                    Coefficient
what happened category / incorrect_operation          0.814435
what happened category / to_be_determined             0.666326
what happened category / other_causes                 0.320360
what happened category / natural_force_damage         0.202986
what happened category / external_interference       -0.060847
what happened category / equipment_failure           -0.101838
what happened category / defect_and_deterioration    -0.897414
what happened category / corrosion_and_c

In [ ]:
## CONCLUSION - none are great predictors of crude oil vs. not
## 3 qualitative cols obtained similar outcomes
## this could mean that crude oil vs. not have similar accident patterns, AKA, inherent properties of crude oil (relative) to other substances don't have an affect on pipeline accidents

## Crude Only

In [966]:
cad_crude = CAD_data_cleaned.copy()
#cad_crude['substance carried'].unique()

# only want to look at crude oil incidents
cad_crude = cad_crude.loc[(cad_crude['substance carried'].str.contains('Crude Oil')) | (cad_crude['released substance type'].str.contains('Crude Oil'))]

# now let's extract the specifications (sour, sweet, heavy, light) into other columns so they're easier to deal with

#cad_crude[['substance carried', 'released substance type']].drop_duplicates()
cad_crude['substance carried'].unique()

## ERROR - i thought it was separate - i.e., crude oil sour, crude oil sweet, but it's actually all or nothing, i.e., crude oil or crude oil sour, sweet, heavy, light
## so this sadly doesn't help our analysis :(

## ONTO dataset B

array(['Crude Oil',
       'Crude Oil, Crude Oil Sour Heavy, Crude Oil Sour Light, Crude Oil Sweet Heavy, Crude Oil Sweet Light',
       'Condensate, Crude Oil, Natural Gas Liquids',
       'Condensate, Crude Oil'], dtype=object)

## unused code

In [ ]:
# ## UPDATE - chem words slims the data down too much. not going to use for now. 

## LOTS OF entries in the qualitative columns - so trying to minimize this by focusing on things that could be related to the substance
## went through for words related to chemistry - temperatures, corrosion, cracking, etc.
## filter first for entries that contain these, make sure lower case
# chem_words = ['corrosion', 'temperature', 'deterioration', 'overheating', 'weather', 'frost', 'fire', 'frozen', 'chemical', 'cracking']

# def contains_chem_words(text, chem_words):
#     if pd.isna(text):
#         return False
#     return any(chem_word in text for chem_word in chem_words)

# Create a boolean mask where at least one of the conditions is True
# mask = test_df.apply(lambda row: contains_chem_words(row['what happened category'], chem_words) or 
#                                  contains_chem_words(row['detailed what happened'], chem_words), axis=1)

# mask = test_df.apply(lambda row: contains_chem_words(row['what happened category'], chem_words), axis=1)

# # Filter the DataFrame
# filtered_df = test_df[mask]

In [ ]:
# want to look at incidents that could have been influenced by the material
# e.g., corrosion/cracking
# hard to determine... leaving this for now
# reasons_df = CAD_data_cleaned.copy()[['Detailed what happened', 'What happened category', 'Detailed why it happened', 'Why it happened category']].drop_duplicates()
# reasons_df['Why it happened category'].unique()

## columns that could be related to material influencing an accident/corrosion

#print(CAD_data_cleaned['Pipeline or Facility Type'].unique())
#CAD_data_cleaned['Rupture'].unique() ## loss of containment and unable to operate
#CAD_data_cleaned['Regulation'].unique()
#print(CAD_data_cleaned['Facility Type'].unique())